# D2

In [ ]:
# import numpy as np
# import os
# import torchaudio
# import re

# data_path = "/mnt/d/PhD/Project_data/Open_source/ALS_data"

# # Define class labels
# LABELS = {
#     "HC": 0,  # Healthy Control (CT files)
#     "ALS": 1   # ALS (PZ files)
# }

# def natural_sort_key(s):
#     return [int(text) if text.isdigit() else text for text in re.split(r'(\d+)', s)]

# def read_data(base_path):
#     s_data = []
#     s_label = []
#     file_list = []  # Store file identifiers

#     # Iterate through all subfolders (phonationA, phonationE, etc.)
#     for folder in os.listdir(base_path):
#         folder_path = os.path.join(base_path, folder)

#         if os.path.isdir(folder_path):
#             print(f"\nProcessing Folder: {folder}")

#             # Get the .wav files sorted naturally
#             wav_files = sorted([f for f in os.listdir(folder_path) if f.lower().endswith('.wav')], key=natural_sort_key)

#             # Process each WAV file separately
#             for wav_file in wav_files:
#                 file_path = os.path.join(folder_path, wav_file)
                
#                 # Load the audio file
#                 waveform, sample_rate = torchaudio.load(file_path)

#                 # Convert stereo to mono if necessary
#                 if waveform.shape[0] > 1:
#                     waveform = waveform.mean(dim=0, keepdim=True)  # Average across channels

#                 # Labeling: files starting with 'CT' are labeled as 'HC', files starting with 'PZ' are labeled as 'ALS'
#                 if wav_file.startswith("CT"):
#                     label = LABELS["HC"]
#                 elif wav_file.startswith("PZ"):
#                     label = LABELS["ALS"]
#                 else:
#                     continue  # Skip files that don't match expected pattern (optional)

#                 # Store the data and labels
#                 s_data.append(waveform.numpy())
#                 s_label.append(label)
#                 file_list.append(f"{folder}/{wav_file}")  # Store filename for reference

#                 print(f"✅ Processed {wav_file} for {folder}, Shape: {waveform.shape}")

#     return s_data, s_label, file_list

# # Load the dataset
# s_data, s_label, file_list = read_data(data_path)



# D3

In [ ]:
# import os
# import librosa
# import re

# LABELS = {
#     "HC": 0,  # Healthy Control
#     "ALS": 1   # ALS
# }

# def natural_sort_key(s):
#     return [int(text) if text.isdigit() else text for text in re.split(r'(\d+)', s)]

# def load_audio_data(base_path):
#     s_data = []
#     s_label = []
#     file_list = []

#     for category in ['ALS', 'HC']:
#         folder_path = os.path.join(base_path, category)
#         if not os.path.exists(folder_path):
#             raise FileNotFoundError(f"Folder '{category}' not found in the specified path.")

#         wav_files = sorted([f for f in os.listdir(folder_path) if f.lower().endswith('.wav')], key=natural_sort_key)

#         for file_name in wav_files:
#             file_path = os.path.join(folder_path, file_name)
#             audio, sr = librosa.load(file_path, sr=None)
#             s_data.append(audio)
#             s_label.append(LABELS[category])
#             file_list.append(f"{category}/{file_name}")
#             print(f"✅ Processed {file_name}, Shape: {audio.shape}")

#     print(f"Total samples: {len(s_data)}")
#     return s_data, s_label, file_list

# # Example usage
# base_path = "/mnt/d/PhD/Project_data/Open_source/Minsk2020_ALS_database-main"
# s_data, s_label, file_list = load_audio_data(base_path)

In [1]:
import gc
import torch

def gpu_freshen():
    # delete your big objects first if they exist
    for name in ["model", "trainer", "optimizer", "scheduler", "batch", "outputs", "logits"]:
        if name in globals():
            del globals()[name]

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        torch.cuda.reset_peak_memory_stats()

gpu_freshen()


# OUR DATA 

In [2]:
import numpy as np
import os
import librosa
import re
import torchaudio
# Define the dataset path
data_path = "/mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/Dataset/ALI_new/All"

# Define class labels
LABELS = {
    "Control": 0,
    "ALSwDysarthria": 1,
    "ALSwoDysarthria": 2
}

def natural_sort_key(s):
    return [int(text) if text.isdigit() else text for text in re.split(r'(\d+)', s)]

def read_data(base_path):
    s_data = []
    s_label = []
    file_list = []  # Store file identifiers

    # Iterate through all categories (Control, ALSwDysarthria, ALSwoDysarthria)
    for category, label in LABELS.items():
        category_path = os.path.join(base_path, category)
        
        # Get the subject folders with natural sorting
        subjects = sorted(
            [f for f in os.listdir(category_path) if os.path.isdir(os.path.join(category_path, f))],
            key=natural_sort_key
        )

        for subject in subjects:
            subject_path = os.path.join(category_path, subject)

            print(f"\nProcessing Subject: {subject} ({category})")

            # Sort files naturally
            wav_files = sorted([f for f in os.listdir(subject_path) if f.endswith('.wav') or f.endswith('.WAV')], key=natural_sort_key)

            # Process each WAV file separately
            for wav_file in wav_files:
                file_path = os.path.join(subject_path, wav_file)
                
                # Load using librosa instead of torchaudio
                try:
                    waveform, sample_rate = librosa.load(file_path, sr=None)
                    # print(f"✅ Processed {wav_file} for {subject}, Shape: {waveform.shape}, Sample rate: {sample_rate}")
                    
                    # Convert to mono if necessary (librosa should already give mono if file is single channel)
                    if len(waveform.shape) > 1 and waveform.shape[0] > 1:
                        waveform = waveform.mean(axis=0)
                    
                    # Store each file separately
                    s_data.append(waveform)
                    s_label.append(label)
                    file_list.append(f"{category}/{subject}/{wav_file}")


                except Exception as e:
                    print(f"❌ Error loading file {wav_file}: {e}")

    return s_data, s_label, file_list
s_data, s_label, file_list = read_data(data_path)



Processing Subject: F1 (Control)

Processing Subject: F2 (Control)

Processing Subject: F3 (Control)

Processing Subject: F4 (Control)

Processing Subject: F5 (Control)

Processing Subject: F6 (Control)

Processing Subject: F7 (Control)

Processing Subject: F8 (Control)

Processing Subject: F9 (Control)

Processing Subject: F10 (Control)

Processing Subject: M1 (Control)

Processing Subject: M2 (Control)

Processing Subject: M3 (Control)

Processing Subject: M4 (Control)

Processing Subject: M5 (Control)

Processing Subject: M6 (Control)

Processing Subject: M7 (Control)

Processing Subject: M8 (Control)

Processing Subject: M9 (Control)

Processing Subject: M10 (Control)

Processing Subject: F1 (ALSwDysarthria)

Processing Subject: F2 (ALSwDysarthria)

Processing Subject: F3 (ALSwDysarthria)

Processing Subject: F4 (ALSwDysarthria)

Processing Subject: F5 (ALSwDysarthria)

Processing Subject: F6 (ALSwDysarthria)

Processing Subject: F7 (ALSwDysarthria)

Processing Subject: F8 (ALSwDy

# Pre-processing 

In [3]:
# Assuming you've already run:
# s_data, s_label, file_list = read_data(data_path)

# Extract subject names (everything before the slash)
subject_ids = set([f.split('/')[0] for f in file_list])

print("Total unique subjects in 'words':", len(subject_ids))
print("Subjects:", sorted(subject_ids))


Total unique subjects in 'words': 3
Subjects: ['ALSwDysarthria', 'ALSwoDysarthria', 'Control']


In [4]:
from collections import Counter

# Count the samples per class label
label_counts = Counter(s_label)

# Print counts with class names
for class_name, class_idx in LABELS.items():
    print(f"{class_name}: {label_counts[class_idx]} samples")


Control: 2212 samples
ALSwDysarthria: 2940 samples
ALSwoDysarthria: 1774 samples


In [5]:

# Load the dataset

import torch
import torchaudio
from transformers import Wav2Vec2Model
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence
import numpy as np

# ---------------------------
# 1. Resample and Preprocess Data
# ---------------------------
def resample_to_16khz(s_data, orig_freq=44100, target_freq=16000):
    resampler = torchaudio.transforms.Resample(orig_freq=orig_freq, new_freq=target_freq)
    # Add channel dim for resampler
    s_data_tensors = [torch.tensor(waveform, dtype=torch.float32).unsqueeze(0) for waveform in s_data]
    # Apply resampling and squeeze back to 1D
    return [resampler(waveform).squeeze(0) for waveform in s_data_tensors]


# Assume s_data is your list of NumPy arrays.
# data is a list of tensors with shape [1, num_samples]
data = resample_to_16khz(s_data)
# Convert each tensor to a 1D tensor and wrap in a dict with key "input_values"
preprocessed_data = []
for waveform in data:
    # waveform shape is [1, num_samples], so squeeze to [num_samples]
    waveform = waveform.squeeze(0)
    preprocessed_data.append({"input_values": waveform})

# 2. Define Collate Function for Variable-Length Inputs
# ---------------------------
def collate_fn(batch):
    """
    Collate function to pad variable-length "input_values" and create an attention mask.
    Each item in `batch` is a dict with key "input_values" (a 1D tensor).
    """
    input_values_list = []
    for item in batch:
        # Expecting item is a dict with "input_values" as a tensor.
        tensor = item["input_values"]
        # Ensure tensor is a 1D tensor.
        if tensor.dim() != 1:
            tensor = tensor.squeeze(0)
        input_values_list.append(tensor)
    
    # Pad the sequences: resulting shape [batch_size, max_seq_length]
    padded_inputs = pad_sequence(input_values_list, batch_first=True, padding_value=0)
    
    # Create attention masks based on original lengths
    attention_masks = [torch.ones(tensor.size(0), dtype=torch.long) for tensor in input_values_list]
    padded_masks = pad_sequence(attention_masks, batch_first=True, padding_value=0)
    
    return {"input_values": padded_inputs, "attention_mask": padded_masks}
data_loader = DataLoader(preprocessed_data, batch_size=4, collate_fn=collate_fn)

for batch in data_loader:
    print(f"Input 'input_values' shape: {batch['input_values'].shape}")
    print(f"Input 'attention_mask' shape: {batch['attention_mask'].shape}")
    break 


/mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Input 'input_values' shape: torch.Size([4, 10968])
Input 'attention_mask' shape: torch.Size([4, 10968])


# Model

In [6]:
import numpy as np

subject_ids = np.asarray(["/".join(str(f).split("/")[:2]) for f in file_list], dtype=object)
print("Unique subjects:", len(np.unique(subject_ids)))
print("Subjects:", sorted(np.unique(subject_ids).tolist()))


Unique subjects: 63
Subjects: ['ALSwDysarthria/F1', 'ALSwDysarthria/F10', 'ALSwDysarthria/F11', 'ALSwDysarthria/F12', 'ALSwDysarthria/F13', 'ALSwDysarthria/F14', 'ALSwDysarthria/F15', 'ALSwDysarthria/F2', 'ALSwDysarthria/F3', 'ALSwDysarthria/F4', 'ALSwDysarthria/F5', 'ALSwDysarthria/F6', 'ALSwDysarthria/F7', 'ALSwDysarthria/F8', 'ALSwDysarthria/F9', 'ALSwDysarthria/M1', 'ALSwDysarthria/M10', 'ALSwDysarthria/M11', 'ALSwDysarthria/M12', 'ALSwDysarthria/M2', 'ALSwDysarthria/M3', 'ALSwDysarthria/M4', 'ALSwDysarthria/M5', 'ALSwDysarthria/M6', 'ALSwDysarthria/M7', 'ALSwDysarthria/M8', 'ALSwDysarthria/M9', 'ALSwoDysarthria/F1', 'ALSwoDysarthria/F2', 'ALSwoDysarthria/M1', 'ALSwoDysarthria/M10', 'ALSwoDysarthria/M11', 'ALSwoDysarthria/M12', 'ALSwoDysarthria/M13', 'ALSwoDysarthria/M14', 'ALSwoDysarthria/M2', 'ALSwoDysarthria/M3', 'ALSwoDysarthria/M4', 'ALSwoDysarthria/M5', 'ALSwoDysarthria/M6', 'ALSwoDysarthria/M7', 'ALSwoDysarthria/M8', 'ALSwoDysarthria/M9', 'Control/F1', 'Control/F10', 'Contro

In [7]:
from transformers import HubertModel
import torch
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


model = HubertModel.from_pretrained("facebook/hubert-xlarge-ls960-ft")


model.to(device)
print(model)


HubertModel(
  (feature_extractor): HubertFeatureEncoder(
    (conv_layers): ModuleList(
      (0): HubertLayerNormConvLayer(
        (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,))
        (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (activation): GELUActivation()
      )
      (1-4): 4 x HubertLayerNormConvLayer(
        (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,))
        (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (activation): GELUActivation()
      )
      (5-6): 2 x HubertLayerNormConvLayer(
        (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,))
        (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (activation): GELUActivation()
      )
    )
  )
  (feature_projection): HubertFeatureProjection(
    (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (projection): Linear(in_features=512, out_features=1280, bias=True)
    (dropout): Dropout(p=

# Data2vec

In [8]:
import numpy as np
import torch

# ===== Data2Vec layer configs (same pooling/concat method; layers adapted to model depth) =====
n_layers = int(getattr(model.config, "num_hidden_layers", 0))
if n_layers <= 0:
    raise ValueError("Could not read model.config.num_hidden_layers")

# hidden_states usually includes an extra "embedding" state at index 0.
# We'll build configs against the available hidden_states length at runtime,
# but use n_layers to choose reasonable depths.
mid = n_layers // 2

# prefer HuBERT-like "stability" around ~9..11 if Data2Vec has it, else fall back near the middle/end safely
stable_start = 9 if n_layers >= 12 else max(mid - 1, 0)
stability = [stable_start, min(stable_start + 1, n_layers - 1), min(stable_start + 2, n_layers - 1)]
middle = [max(mid - 1, 0), min(mid, n_layers - 1), min(mid + 1, n_layers - 1)]

LAYER_CONFIGS = {
    "stability": stability,
    "acoustic": [0, 1, 2] if n_layers >= 3 else list(range(n_layers)),
    "middle": middle,
    "last3": [-3, -2, -1],                 # always "last 3" regardless of depth
    "multi-depth": [0, stability[0], middle[-1], -1],
    "all": "ALL",                          # resolved dynamically per forward pass
}

def _resolve_layers(spec, *, n_states: int) -> list[int]:
    # spec can be "ALL" or list of indices (supports negatives)
    if spec == "ALL":
        return list(range(n_states))
    idxs = []
    for x in spec:
        i = int(x)
        if i < 0:
            i = n_states + i
        if not (0 <= i < n_states):
            raise ValueError(f"Layer index out of range: {x} resolved to {i} with n_states={n_states}")
        idxs.append(i)
    return idxs

def extract_and_save_features(layer_sets=LAYER_CONFIGS):
    """
    Returns dictionary of pooled+concatenated features per layer-set (model frozen).
    """
    all_features = {name: [] for name in layer_sets}

    for batch in data_loader:
        input_values = batch["input_values"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        with torch.no_grad():
            outputs = model(input_values, attention_mask=attention_mask, output_hidden_states=True)

        hidden_states = outputs.hidden_states
        n_states = len(hidden_states)

        for name, spec in layer_sets.items():
            layers = _resolve_layers(spec, n_states=n_states)

            pooled_features = []
            for layer in layers:
                feature = hidden_states[layer]
                seq_len = feature.size(1)
                adj_mask = attention_mask[:, :seq_len].unsqueeze(-1).float()

                masked_feature = feature * adj_mask
                valid_counts = adj_mask.sum(dim=1).clamp(min=1e-9)
                mean_pooled = masked_feature.sum(dim=1) / valid_counts
                pooled_features.append(mean_pooled)

            final_features = torch.cat(pooled_features, dim=-1).cpu().numpy()
            all_features[name].append(final_features)

    for name in all_features:
        all_features[name] = np.concatenate(all_features[name], axis=0)

    return all_features

all_features = extract_and_save_features()
print("Extracted feature sets:", {k: v.shape for k, v in all_features.items()})


Extracted feature sets: {'stability': (6926, 3840), 'acoustic': (6926, 3840), 'middle': (6926, 3840), 'last3': (6926, 3840), 'multi-depth': (6926, 5120), 'all': (6926, 62720)}


In [9]:
# import numpy as np
# import torch

# # Define layer configurations
# LAYER_CONFIGS = {
#     "stability": [9, 10, 11],     # Your original
#     "acoustic": [0, 1, 2],      # Early-stage
#     "middle": [22, 23, 24],  # Disease progression
#     "last3": [45, 46, 47],      # Late-stage
#     "multi-depth": [0, 5, 12, 47],
#     "all": list(range(0, 48)),  
# }

# def extract_and_save_features(layer_sets=LAYER_CONFIGS):
#     """
#     Returns dictionary of features and saves them to disk
#     """
#     # Dictionary to store features
#     all_features = {name: [] for name in layer_sets}
    
#     for batch in data_loader:
#         input_values = batch["input_values"].to(device)
#         attention_mask = batch["attention_mask"].to(device)
        
#         # Single forward pass
#         with torch.no_grad():
#             outputs = model(input_values, attention_mask=attention_mask, output_hidden_states=True)
        
#         # Process each layer configuration
#         for name, layers in layer_sets.items():
#             pooled_features = []
#             for layer in layers:
#                 feature = outputs.hidden_states[layer]
#                 seq_len = feature.size(1)
#                 adj_mask = attention_mask[:, :seq_len].unsqueeze(-1).float()
                
#                 # Masked mean pooling
#                 masked_feature = feature * adj_mask
#                 valid_counts = adj_mask.sum(dim=1).clamp(min=1e-9)
#                 mean_pooled = masked_feature.sum(dim=1) / valid_counts
#                 pooled_features.append(mean_pooled)
            
#             # Concatenate and store
#             final_features = torch.cat(pooled_features, dim=-1).cpu().numpy()
#             all_features[name].append(final_features)
    
#     # Concatenate all batches for each feature set
#     for name in all_features:
#         all_features[name] = np.concatenate(all_features[name], axis=0)

    
#     return all_features  # This was missing in your original code

# # Now this will work correctly
# all_features = extract_and_save_features()



In [10]:
from pathlib import Path
out_dir = Path('/mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/code/Hubert')
out_dir.mkdir(parents=True, exist_ok=True)

last3 = all_features["last3"]           # shape: [n_samples, 3840]
middle3 = all_features["middle"]       # shape: [n_samples, 3840]  
acoustic = all_features["acoustic"]     # shape: [n_samples, 3840]
stability = all_features["stability"]   # shape: [n_samples, 3840]
full_spec = all_features["multi-depth"]
all_layers = all_features["all"]
# Only these lines changed (xlarge_hubert_* → xwav2vec2_*)
np.save(out_dir / "xlarge_hubert_last3.npy", last3)
np.save(out_dir / "xlarge_hubert_middle3.npy", middle3)
np.save(out_dir / "xlarge_hubert_acoustic.npy", acoustic)
np.save(out_dir / "xlarge_hubert_stability.npy", stability)
np.save(out_dir / "xlarge_hubert_fullspec.npy", full_spec)
np.save(out_dir / "xlarge_hubert_all.npy", all_layers)
np.save(out_dir / "labels.npy", s_label)  # Save labels once (unchanged)

# Save file_list + per-utterance subject IDs (for leakage-safe splits)
np.save(out_dir / "file_list.npy", np.asarray(file_list, dtype=object))
subject_ids = np.asarray(["/".join(f.split("/")[:2]) for f in file_list], dtype=object)
np.save(out_dir / "subject_ids.npy", subject_ids)

# # Print statements remain identical
print(f"last3     shape: {last3.shape}")


last3     shape: (6926, 3840)


In [ ]:
# # last3 = all_features["last3"]           # shape: [n_samples, 3840]
# middle3 = all_features["middle"]       # shape: [n_samples, 3840]  
# # acoustic = all_features["acoustic"]     # shape: [n_samples, 3840]
# stability = all_features["stability"]   # shape: [n_samples, 3840]
# full_spec = all_features["full_spectrum"]
# all_layers = all_features["all"]


# # np.save("xlarge_hubert_last3.npy", last3)
# np.save("xlarge_hubert_middle3.npy", middle3)
# # np.save("xlarge_hubert_acoustic.npy", acoustic)
# np.save("xlarge_hubert_stability.npy", stability)
# np.save("xlarge_hubert_fullspec.npy", full_spec)
# np.save("xlarge_hubert_all.npy", all_layers)
# np.save("labels.npy", s_label)  # Save labels once
#    # shape: [n_samples, 3840]
# print(f"last3     shape: {last3.shape}")
# print(f"middle3   shape: {middle3.shape}")
# print(f"acoustic  shape: {acoustic.shape}")
# print(f"stability shape: {stability.shape}")
# print(f"full_spec shape: {full_spec.shape}")
